# Widgets and an editable table

Widgets in the `morris_lisp` kernel: sliders and buttons that call Lisp, and a grid to edit a table in. Needs `ipywidgets` and `ipydatagrid`. The manual's "Widgets in a notebook" has all of it.

## A mortgage payment, as sliders move

A function that a widget calls has no cell to write to, so `:output out` says where what it displays goes.

In [ ]:
(define out (widget-output))

(define (show-payment)
  (widget-clear out)
  (widget-capture out
    (lambda ()
      (display (format "Monthly payment: {:,.2f}\n"
                       (payment (/ (widget-get rate "value") 1200)
                                (* 12 (widget-get years "value"))
                                (widget-get balance "value")))))))

(define balance (widget-slider 50000 1000000 :value 300000 :step 10000 :description "balance" :on-change (lambda (v) (show-payment))))
(define rate    (widget-slider 1 10 :value 6.5 :step 0.125 :description "rate %" :on-change (lambda (v) (show-payment))))
(define years   (widget-slider 5 40 :value 30 :description "years" :on-change (lambda (v) (show-payment))))
(show-payment)
(widget-column balance rate years out)

## Editing a table

`table-grid` makes a grid of a table. Double-click a cell to change it; click a heading to sort. `grid-on-edit` hears each change, and `grid-table` gives the table as it is now.

In [ ]:
(define loans
  (make-table "id"      (vector "A-1" "A-2" "A-3")
              "balance" (vector 250000.0 410000.0 175500.0)
              "rate"    (vector 0.055 0.0625 0.0475)
              "state"   (vector "NY" "CA" "TX")))

(define grid (table-grid loans))
(define log (widget-output))
(grid-on-edit grid
              (lambda (column row value) (display (format "{} of loan {} is now {}\n" column row value)))
              :output log)
(widget-column grid log)

Change some cells, then press the button: it shows the edited table, and the total balance of it.

In [ ]:
(define report (widget-output))

(define (show-edits)
  (widget-clear report)
  (widget-capture report
    (lambda ()
      (let ((now (grid-table grid)))
        (display-table now '(("balance" ",.2f") ("rate" ".3%")))
        (display (format "total balance {:,.2f}\n" (vector-sum (table-column now "balance"))))))))

(widget-column (widget-button "Show the table" (lambda () (show-edits)) :button-style "primary") report)

## What to do with a tranche

Click a tranche in the grid: a panel opens below it, with a tab for each thing that can be done with a tranche, the fields that thing needs, and a **Go** button. `widget-actions` makes the panel; `grid-on-click` hears the click; `widget-actions-heading`, `-select` and `-set!` fit the panel to the tranche that was clicked. What the button's function displays appears under the tabs.

In [ ]:
(define tranches
  (make-table "name"    (vector "A-1" "A-2" "F-1" "P-1")
              "type"    (vector "sequential" "sequential" "floater" "PAC")
              "balance" (vector 120000000 80000000 50000000 100000000)
              "coupon"  (vector 0.045 0.05 0.062 0.05)))

(define (tranche-of row column) (vector-ref (table-column tranches column) row))
(define chosen '())                      ; the row of the tranche that was clicked

(define panel
  (widget-actions
    (list (list "Floater cap" "Limits how high the coupon of a floating-rate tranche can go."
                (list "cap"   'number "Cap %" 8.0)
                (list "floor" 'number "Floor %" 0.0))
          (list "PAC bands" "The prepayment speeds (PSA) that the tranche is protected over."
                (list "low"  'integer "Lower band" 100)
                (list "high" 'integer "Upper band" 300))
          (list "Split" "Cut the tranche into parts."
                (list "parts" 'integer "Parts" 2)
                (list "even"  'checkbox "Equal sizes" #t)))
    :heading "Click a tranche."
    :on-go (lambda (action answers)
             (display (format "Tranche {}: {} {}\n" (tranche-of chosen "name") action answers)))))

(define (choose-tranche column row value)
  (set! chosen row)
  (let ((type (tranche-of row "type")))
    (widget-actions-heading panel
      (format "<b>Tranche {}</b>: {}, balance {:,.0f}, coupon {:.2%}"
              (tranche-of row "name") type (tranche-of row "balance") (tranche-of row "coupon")))
    (widget-actions-select panel (cond ((equal? type "floater") "Floater cap")
                                       ((equal? type "PAC") "PAC bands")
                                       (else "Split")))
    (widget-actions-set! panel "Floater cap" "cap" (* 100 (+ 0.03 (tranche-of row "coupon"))))
    (widget-visible panel #t)))

(define tranche-grid (table-grid tranches :editable #f))
(grid-on-click tranche-grid choose-tranche)
(widget-visible panel #f)
(widget-column tranche-grid panel)